<a href="https://colab.research.google.com/github/GlobalFishingWatch/gfw-api-python-client/blob/develop/notebooks/workflow-guides/workflow-03-analyze-fleet-in-ghanaian-eez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analyze a fleet in Ghanaian EEZ

This guide provides detailed instructions to on how to use the [gfw-api-python-client](https://github.com/GlobalFishingWatch/gfw-api-python-client) to **Monitor a Fleet (a group of vessels) of Tuna Longliners in [Ghanaian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8400) region for Compliance** using **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)**, **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)**, and **[Events API](https://globalfishingwatch.org/our-apis/documentation#events-api)**.

**Note:** See the [Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset), [Data Caveats](https://globalfishingwatch.org/our-apis/documentation#data-caveat), and [Terms of Use](https://globalfishingwatch.org/our-apis/documentation#terms-of-use) pages in the [GFW API documentation](https://globalfishingwatch.org/our-apis/documentation#introduction) for details on GFW data, API licenses, and rate limits.

## Prerequisites

Before using the `gfw-api-python-client`, ensure it is installed (see the [Getting Started](https://globalfishingwatch.github.io/gfw-api-python-client/getting-started.html) guide) and that you have obtained an API access token from the [Global Fishing Watch API portal](https://globalfishingwatch.org/our-apis/tokens).

## Installation

The `gfw-api-python-client` can be easily installed using pip:

In [1]:
# %pip install gfw-api-python-client

## Usage

Import and use `gfw-api-python-client` in your Python codes

In [2]:
import os

import pandas as pd

import gfwapiclient as gfw

In [3]:
try:
    from google.colab import userdata

    access_token = userdata.get("GFW_API_ACCESS_TOKEN")
except Exception:
    access_token = os.environ.get("GFW_API_ACCESS_TOKEN")

access_token = access_token or "<PASTE_YOUR_GFW_API_ACCESS_TOKEN_HERE>"

In [4]:
gfw_client = gfw.Client(
    access_token=access_token,
)

## Introduction

**Use Case: Monitoring a Fleet of Tuna Longliners for Compliance**

Kwame is a fisheries compliance officer in Ghana, responsible for monitoring a fleet of tuna longliners operating within **[Ghanaian Exclusive Economic Zone (EEZ)](https://www.marineregions.org/gazetteer.php?p=details&id=8400)**. His goal is to:

1. Track apparent fishing effort for **longliners** over the last 12 months.
2. Identify potential vessels in this fleet, their operational patterns, and their activity levels.
3. Retrieve vessel details, including **flag state**, **ownership history**, and **authorizations**.
4. Analyze events such as **port visits** and **encounters (potential transshipment)** activities.

**APIs Used:**
️
1. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** – Retrieve apparent fishing effort grouped by vessel ID in Ghanaian EEZ.
2. **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** – Get vessel identity, ownership, and compliance details.
3. **[Events API](https://globalfishingwatch.org/our-apis/documentation#events-api)** – Identify port visits and potential transshipment activities for vessels in the fleet.

**Important:** In order to avoid any misinterpretation of **GFW data**, please refer to our official **data caveats** documentations:
- [Apparent fishing effort](https://globalfishingwatch.org/dataset-and-code-fishing-effort/) 
- [Exclusive economic zone boundaries definition](https://globalfishingwatch.org/our-apis/documentation#exclusive-economic-zone-boundaries-definition)
- [Vessel ID](https://globalfishingwatch.org/our-apis/documentation#vessel-id)
- [Vessel API - Vessel identity information](https://globalfishingwatch.org/our-apis/documentation#vessel-api-vessel-identity-information)

**Important Caveats:**

1. The [4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api) only supports **one active report per user at a time**.
2. **Sending multiple requests simultaneously** results in a **429 Too Many Requests** error.
3. If a report takes over **100 seconds** to generate, it may return a **524 Gateway Timeout** error.

## Step 0: Identify the Region of Interest (ROI) - Ghanaian EEZ

Before making API requests, Kwame must specify the geographic area for analysis using a **Region ID**:

**Options to Define the Region:**

1. **Using Region ID** - Each EEZ has a unique ID in the **[public-eez-areas](https://globalfishingwatch.org/our-apis/documentation#regions)** dataset.
2. **Custom Geometries** - Users can define a custom area using GeoJSON.
   
For **[Ghanaian EEZ, the region ID is 8400](https://www.marineregions.org/gazetteer.php?p=details&id=8400)** (public-eez-areas dataset).

## Step 1: Retrieve Fishing Effort in Ghanaian EEZ

Kwame **first queries** the **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** to get **fishing effort for all vessels**, grouping them by **gear type** in **[Ghanaian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8400)**. Please [learn more about apparent fishing effort here](https://globalfishingwatch.org/our-apis/documentation#ais-apparent-fishing-effort) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#apparent-fishing-effort).

**Filters Used:**

1. **[Region ID](https://globalfishingwatch.org/our-apis/documentation#regions)** - 8400 Ghanaian EEZ
2. **[Date Range](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Last 12 Months
3. **[Grouped By](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Gear Type

In [5]:
step_1_report_result = await gfw_client.fourwings.create_fishing_effort_report(
    spatial_resolution="LOW",
    group_by="GEARTYPE",
    temporal_resolution="ENTIRE",
    start_date="2024-01-01",
    end_date="2025-01-01",
    spatial_aggregation=True,
    region={
        "dataset": "public-eez-areas",
        "id": "8400",
    },
)

In [6]:
step_1_report_df = step_1_report_result.df()

In [7]:
step_1_report_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   date                     9 non-null      object 
 1   detections               0 non-null      object 
 2   flag                     0 non-null      object 
 3   gear_type                9 non-null      object 
 4   hours                    9 non-null      float64
 5   vessel_ids               9 non-null      int64  
 6   vessel_id                0 non-null      object 
 7   vessel_type              0 non-null      object 
 8   entry_timestamp          0 non-null      object 
 9   exit_timestamp           0 non-null      object 
 10  first_transmission_date  0 non-null      object 
 11  last_transmission_date   0 non-null      object 
 12  imo                      0 non-null      object 
 13  mmsi                     0 non-null      object 
 14  call_sign                0 non

In [8]:
step_1_report_df[["flag", "gear_type", "hours", "vessel_ids"]].head()

,flag,gear_type,hours,vessel_ids
0,None,drifting_longlines,593.138333,3
1,None,purse_seines,6.340556,1
2,None,pole_and_line,3481.509167,5
3,None,other_purse_seines,26.581111,1
4,None,fishing,20929.563333,21


In [9]:
step_1_agg_report_df = (
    step_1_report_df.groupby(["gear_type"], as_index=False)
    .agg(hours=("hours", "sum"), vessel_ids=("vessel_ids", "sum"))
    .sort_values(by="hours", ascending=False)
)

In [10]:
step_1_agg_report_df

,gear_type,hours,vessel_ids
7,trawlers,46704.797222,26
3,inconclusive,30496.173611,27
1,fishing,20929.563333,21
8,tuna_purse_seines,5146.623889,23
5,pole_and_line,3481.509167,5
0,drifting_longlines,593.138333,3
4,other_purse_seines,26.581111,1
6,purse_seines,6.340556,1
2,fixed_gear,0.163611,1


### What We have Learned from Step 1

1. Kwame now has apparent fishing effort data for multiple gear types.
2. There are **potential 3 vessels** operating as a **longliners (i.e., drifting_longlines)** in **[Ghanaian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8400)** with `593.138333 hours` logged.

## Step 2: Retrieve Vessel IDs for Longliners

Kwame refines his **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** request to group by **vessel ID**, and filtering only for **longliners** in **[Ghanaian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8400)**.

**Filters Used:**

1. **[Region ID](https://globalfishingwatch.org/our-apis/documentation#regions)** - [8400 Ghanaian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8400)
2. **[Date Range](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Last 12 Months
3. **[Grouped By](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Vessel ID 

**Why Use group-by=VESSEL_ID?**

Grouping by **VESSEL_ID** allows **individual vessel identification** in the response. This is crucial for **tracking vessel activity** and, more importantly, linking each detected vessel to the **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** in the next step. By structuring the query this way, we can fetch vessel details such as **flag, name, and ownership records** in **Step 3 below**.


In [11]:
step_2_report_result = await gfw_client.fourwings.create_fishing_effort_report(
    spatial_resolution="LOW",
    group_by="VESSEL_ID",
    temporal_resolution="ENTIRE",
    filters=["geartype in ('drifting_longlines')"],
    start_date="2024-01-01",
    end_date="2025-01-01",
    spatial_aggregation=True,
    region={
        "dataset": "public-eez-areas",
        "id": "8400",
    },
)

In [12]:
step_2_report_df = step_2_report_result.df()

In [13]:
step_2_report_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype              
---  ------                   --------------  -----              
 0   date                     3 non-null      object             
 1   detections               0 non-null      object             
 2   flag                     3 non-null      object             
 3   gear_type                3 non-null      object             
 4   hours                    3 non-null      float64            
 5   vessel_ids               0 non-null      object             
 6   vessel_id                3 non-null      object             
 7   vessel_type              3 non-null      object             
 8   entry_timestamp          3 non-null      datetime64[ns, UTC]
 9   exit_timestamp           3 non-null      datetime64[ns, UTC]
 10  first_transmission_date  3 non-null      datetime64[ns, UTC]
 11  last_transmission_date   3 non-null 

In [14]:
step_2_report_df[["flag", "gear_type", "hours", "mmsi", "ship_name"]].head()

,flag,gear_type,hours,mmsi,ship_name
0,CHN,DRIFTING_LONGLINES,2.750556,412331032,
1,JPN,DRIFTING_LONGLINES,588.879722,431100690,SENSHU MARU NO.3
2,TWN,DRIFTING_LONGLINES,1.508056,416007496,HUNG CHUAN SHUN


### What We have Learned from Step 2

1. Kwame identifies `3 vessels` operating as longliners within **[Ghanaian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8400)**.
2. The vessel `(mmsi: 431100690, ship_name: SENSHU MARU NO.3)` shows significant activity with `588.879722 hours` logged.
3. Other vessels `(mmsi: 416007496, ship_name: HUNG CHUAN SHUN)` and `(mmsi: 412331032)` shows apparent fishing effort over a short duration.
4. This response is based on **AIS self-reported** data and should be further validated.


## Step 3: Retrieve Vessel Details Using the Vessels API

Kwame queries the **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** to **get detailed vessel identity and ownership records**. Please [learn more about Vessels API here](https://globalfishingwatch.org/our-apis/documentation#vessels-api) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#vessel-api-vessel-identity-information).

**Filters Used:**

1. **Vessel IDs** from 4Wings API, **Step 2 above**.
2. **[Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset)** - `public-global-vessel-identity:latest`.
3. **[Includes](https://globalfishingwatch.org/our-apis/documentation#get-vessels-by-ids-url-parameters)** - `POTENTIAL_RELATED_SELF_REPORTED_INFO`.

**Note:** Vessels may change identifiers over time, such as their `Maritime Mobile Service Identity (MMSI)`,` International Maritime Organization (IMO) number)`, `call sign`, or even their `name`. These changes can occur due to `re-registration`, `changes in ownership`, or other `operational reasons` within the `AIS transponder`. Parameter (`includes = POTENTIAL_RELATED_SELF_REPORTED_INFO`) helps group all **vessel ids** that are **potentially related** as part of the **same physical vessel** based on publicly available registry information.

In [15]:
step_2_vessel_ids = list(step_2_report_df["vessel_id"].unique())

In [16]:
step_2_vessel_ids

['f37ebdc1b-be44-0740-7904-49397360e29d',
 'b1dad8628-8c9c-2ee7-258b-3d8fb747f1c8',
 '60f7bb972-2c90-4553-650b-23c38f9521bf']

In [17]:
step_3_vessels_result = await gfw_client.vessels.get_vessels_by_ids(
    ids=step_2_vessel_ids,
)

In [18]:
step_3_vessels_df = step_3_vessels_result.df()

In [19]:
step_3_vessels_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 7 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   dataset                         3 non-null      object
 1   registry_info_total_records     3 non-null      int64 
 2   registry_info                   3 non-null      object
 3   registry_owners                 3 non-null      object
 4   registry_public_authorizations  3 non-null      object
 5   combined_sources_info           3 non-null      object
 6   self_reported_info              3 non-null      object
dtypes: int64(1), object(6)
memory usage: 300.0+ bytes


**Understanding Vessel Details Response Data**

- **registryInfoTotalRecords** – This represents the **number of registry records** found for the vessels.
- **registryInfo** – Contains **public registry data**. This data is sourced from official **vessel registries**.
- **registryOwners** – Lists the **registered owners** of the vessel based on public sources.
- **registryPublicAuthorizations** – Represents known **fishing authorizations** from public sources. Users should verify against national registries and RFMO records for additional context.
- **combinedSourcesInfo** – Provides inferred data from multiple sources, including. This is not explicitly reported by vessels but determined through **GFW's classification methods**.
- **selfReportedInfo** – Contains **AIS self-reported** data, including `MMSI`, `ship name`, and `flag` as broadcast by the **vessel itself**. Self-reported data may not always align with registry data and should be cross-checked.

In [20]:
step_3_vessels_df[["registry_info", "registry_owners", "self_reported_info"]]

,registry_info,registry_owners,self_reported_info
0,"[{'id': '0b0dec977c1aad4ae6652c4076572cc7', 's...","[{'name': 'TOMIOKA FISHERIES', 'flag': 'JPN', ...",[{'id': 'c86d138c5-5d51-3620-fe01-a9e0ed37fb91...
1,[],[],[{'id': 'f37ebdc1b-be44-0740-7904-49397360e29d...
2,"[{'id': '02eda7d2da02943eecd48813fb7d562a', 's...","[{'name': 'HER RONG SHUN FISHERIES', 'flag': '...",[{'id': '60f7bb972-2c90-4553-650b-23c38f9521bf...


### Explore Vessels Registry Info

In [21]:
step_3_registry_info_df = pd.json_normalize(
    step_3_vessels_df["registry_info"].explode()
)
step_3_registry_info_df = step_3_registry_info_df.dropna()

In [22]:
step_3_registry_info_df[
    ["ssvid", "flag", "ship_name", "n_ship_name", "gear_types", "source_code"]
]

,ssvid,flag,ship_name,n_ship_name,gear_types,source_code
0,431100690,JPN,SENSHU MARU NO.3,SENSHUMARU3,[DRIFTING_LONGLINES],"[CCSBT, GFCM, IATTC, ICCAT, IMO, IOTC, OPRT, R..."
2,416007496,TWN,HUNG CHUAN SHUN,HUNGCHUANSHUN,[DRIFTING_LONGLINES],"[ICCAT, IMO, ISSF, OPRT, TMT_ICCAT, TMT_OTHER_..."


### Explore Registry Owners

In [23]:
step_3_registry_owners_df = pd.json_normalize(
    step_3_vessels_df["registry_owners"].explode()
)
step_3_registry_owners_df = step_3_registry_owners_df.dropna()

In [24]:
step_3_registry_owners_df[["ssvid", "flag", "name", "source_code"]]

,ssvid,flag,name,source_code
0,431100690,JPN,TOMIOKA FISHERIES,"[TMT_CCSBT, TMT_IATTC, TMT_ICCAT, TMT_OTHER_OF..."
1,431100690,JPN,TOMIOKA,"[TMT_CCSBT, TMT_IATTC, TMT_ICCAT, TMT_OTHER_OF..."
2,431100690,JPN,YAMAMOTO YUUKI,"[TMT_CCSBT, TMT_IATTC, TMT_ICCAT, TMT_OTHER_OF..."
3,431100690,JPN,YAMAMOTO HIROKI,"[TMT_CCSBT, TMT_IATTC, TMT_ICCAT, TMT_OTHER_OF..."
5,416007496,TWN,HER RONG SHUN FISHERIES,"[TMT_ICCAT, TMT_OTHER_OFFICIAL]"


### Explore Vessels Self Reported Info

In [25]:
step_3_self_reported_info_df = pd.json_normalize(
    step_3_vessels_df["self_reported_info"].explode()
)

In [26]:
step_3_self_reported_info_df[
    ["ssvid", "flag", "ship_name", "n_ship_name", "source_code"]
]

,ssvid,flag,ship_name,n_ship_name,source_code
0,431100690,JPN,None,None,[AIS]
1,431100690,JPN,SENSHU MARU NO.3,SENSHUMARU3,[AIS]
2,431100690,JPN,SENSHU MARU NO3,SENSHUMARU3,[AIS]
3,412331032,CHN,None,None,[AIS]
4,416007496,TWN,HUNG CHUAN SHUN,HUNGCHUANSHUN,[AIS]


### What We have Learned from Step 3

- The vessel `mmsi/ssvid: 412331032` appears to be a drifting longliner flagged under China.
- No public registry data is found for this vessel.
- The vessel's identity information is based on `AIS self-reported data`, which may not always align with official registries.
- The vessel appears to have been active since `2014`, based on `self-reported AIS records`.
- This vessel's data needs further validation against official public sources

## Step 4: Detect Fleet Activity (Port Visits)

Now that Kwame has identified vessels in the fleet, he examines their activity further by querying the **[Events API](https://globalfishingwatch.org/our-apis/documentation#events-api)**. This allows him to detect `port visits`, `encounters (Potential Transshipment)` and `apparent fishing activity` based on vessel movement patterns. Please [learn more about Events API here](https://globalfishingwatch.org/our-apis/documentation#events-api) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#how-are-the-events-estimated).

**Filters Used:**

1. **Vessel ID** from 4Wings API
2. **[Event Types](https://globalfishingwatch.org/our-apis/documentation#events-post-body-parameters)** - Port visits, encounters (potential transshipment), and fishing events.
3. **Time Range** - Last 6 months.
4. **[Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset)**:
   - `public-global-port-visits-events::latest` (Port Visits)
   - `public-global-encounters-events:latest` (Encounters between vessels)
   - `public-global-fishing-events:latest` (Fishing activity)
5. **[Encounter Types](https://globalfishingwatch.org/our-apis/documentation#events-post-body-parameters)** - FISHING-FISHING

In [27]:
step_4_events_result = await gfw_client.events.get_all_events(
    datasets=[
        "public-global-encounters-events:latest",
        "public-global-fishing-events:latest",
        "public-global-port-visits-events:latest",
    ],
    vessels=step_2_vessel_ids,
    types=["ENCOUNTER", "FISHING", "PORT_VISIT"],
    start_date="2024-08-01",
    end_date="2025-01-31",
    encounter_types=["FISHING-FISHING"],
)

In [28]:
step_4_events_df = step_4_events_result.df()

In [29]:
step_4_events_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   start         418 non-null    datetime64[ns, UTC]
 1   end           418 non-null    datetime64[ns, UTC]
 2   id            418 non-null    object             
 3   type          418 non-null    object             
 4   position      418 non-null    object             
 5   regions       418 non-null    object             
 6   bounding_box  418 non-null    object             
 7   distances     418 non-null    object             
 8   vessel        418 non-null    object             
 9   encounter     0 non-null      object             
 10  fishing       414 non-null    object             
 11  gap           0 non-null      object             
 12  loitering     0 non-null      object             
 13  port_visit    4 non-null      object             
dtypes: datetim

In [30]:
step_4_events_df["type"].value_counts()

type
fishing       414
port_visit      4
Name: count, dtype: int64

### Explore Apparent Fishing Events

In [31]:
step_4_fishing_events_df = step_4_events_df[step_4_events_df["fishing"].notna()]

In [32]:
step_4_fishing_df = pd.concat(
    [
        pd.json_normalize(step_4_fishing_events_df["vessel"], sep="_"),
        pd.json_normalize(step_4_fishing_events_df["fishing"], sep="_"),
    ],
    axis=1,
)

In [33]:
step_4_fishing_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 414 entries, 0 to 413
Data columns (total 12 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   id                                  414 non-null    object 
 1   name                                389 non-null    object 
 2   ssvid                               414 non-null    object 
 3   flag                                414 non-null    object 
 4   type                                414 non-null    object 
 5   public_authorizations               414 non-null    object 
 6   nextPort                            0 non-null      object 
 7   total_distance_km                   414 non-null    float64
 8   average_speed_knots                 414 non-null    float64
 9   average_duration_hours              0 non-null      object 
 10  potential_risk                      414 non-null    bool   
 11  vessel_public_authorization_status  414 non-n

In [34]:
step_4_fishing_df[
    [
        "name",
        "ssvid",
        "total_distance_km",
        "average_speed_knots",
    ]
]

,name,ssvid,total_distance_km,average_speed_knots
0,HUNG CHUAN SHUN,416007496,44.026882,3.808824
1,HUNG CHUAN SHUN,416007496,9.353500,8.200000
2,HUNG CHUAN SHUN,416007496,9.868050,5.494444
3,HUNG CHUAN SHUN,416007496,10.955310,8.555556
4,HUNG CHUAN SHUN,416007496,9.630687,8.942857
...,...,...,...,...
409,HUNG CHUAN SHUN,416007496,101.402263,4.041791
410,HUNG CHUAN SHUN,416007496,115.642159,4.743357
411,HUNG CHUAN SHUN,416007496,100.019456,4.402564
412,HUNG CHUAN SHUN,416007496,155.923138,4.975000


In [35]:
step_4_fishing_df["ssvid"].value_counts()

ssvid
416007496    363
431100690     26
412331032     25
Name: count, dtype: int64

### Explore Port Visit Events

In [36]:
step_4_port_visit_events_df = step_4_events_df[step_4_events_df["port_visit"].notna()]

In [37]:
step_4_port_visits_df = pd.concat(
    [
        pd.json_normalize(step_4_port_visit_events_df["vessel"], sep="_"),
        pd.json_normalize(step_4_port_visit_events_df["port_visit"], sep="_"),
    ],
    axis=1,
)

In [38]:
step_4_port_visits_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 37 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   id                                             4 non-null      object 
 1   name                                           3 non-null      object 
 2   ssvid                                          4 non-null      object 
 3   flag                                           4 non-null      object 
 4   type                                           4 non-null      object 
 5   public_authorizations                          4 non-null      object 
 6   nextPort                                       0 non-null      object 
 7   visit_id                                       4 non-null      object 
 8   confidence                                     4 non-null      object 
 9   duration_hrs                                   4 non-null 

In [39]:
step_4_port_visits_df[
    [
        "name",
        "ssvid",
        "confidence",
        "start_anchorage_name",
        "intermediate_anchorage_name",
        "end_anchorage_name",
    ]
]

,name,ssvid,confidence,start_anchorage_name,intermediate_anchorage_name,end_anchorage_name
0,SENSHU MARU NO.3,431100690,4,TEMA,TEMA,TEMA
1,HUNG CHUAN SHUN,416007496,4,TEMA,TEMA,TEMA
2,SENSHU MARU NO.3,431100690,4,TEMA,TEMA,TEMA
3,None,412331032,4,DAKAR,DAKAR,DAKAR


### What We’ve Learned from Step 4

- `4: port visits`, `0: encounters`, and `414: fishing` events were found for the queried vessels in the given date range.
- Some events were missed due to **AIS data coverage gaps**.
- Different filters may need to be applied to refine results.

**Caveats & Considerations**

- **A lack of recorded encounters or any other events does not confirm the absence of such activities**—AIS coverage, reporting behavior, and dataset updates can impact results.
- **Further investigation may be required**, including manual validation using historical data or consulting additional sources.

## Summary of API Flow

1. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** - Identify fishing effort by **gear type** in Ghanaian EEZ.
2. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** - Retrieve vessel IDs for potential **longliners**.
3. **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** - Fetch detailed **vessel identity** & **ownership**.
4. **[Events API](https://globalfishingwatch.org/our-apis/documentation#events-api)** - Attempt to detect fleet activity (**port visits**, **encounters**, and **apparent fishing** events)